# Predicting Real-World Arm Use After Stroke

**Author:** Asad NADEEM  
**Course:** Python-R-Git  
**Program:** Master STAPS — Sciences Technologies Mouvement  
**Lab:** EuroMov Digital Health in Motion — University of Montpellier  
**Supervisors:** Karima BAKHTI, Makii MUTHALIB, Nicolas SUTTON-CHARANI  

**GitHub:** https://github.com/asadazizmughal/predicting-arm-use-stroke

---

## Scientific Question

Can clinical motor and functional tests predict how much stroke patients actually use their paretic (weak) arm in daily life at home?

## Notebook overview

1. **Setup** — import libraries, create output folder
2. **Load clinical data** — 30 patients with FMUE, WMFT, BBT, Barthel, PANU scores
3. **Load accelerometry data** — 7-day wrist-worn sensor counts
4. **Compute funcUseRatio** — target variable
5. **Merge datasets**
6. **LOOCV modeling** with in-fold imputation (Prof. Dray's recommendation)
7. **Compare models** — Linear Regression, Ridge, Lasso
8. **Feature importance & discrepancy analysis**

---

## 1. Setup

We import all libraries we will use throughout the notebook and set plot styles. The `results/` folder is created automatically if it doesn't exist — this makes the project portable (works on any computer).

In [1]:
# Library imports for the full ML pipeline
# Data handling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
import openpyxl
import os

# Machine learning (scikit-learn)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Make sure results folder exists
os.makedirs('results', exist_ok=True)

print("Setup complete — libraries loaded, results folder ready.")

Setup complete — libraries loaded, results folder ready.


## 2. Load Clinical Data

The clinical data comes from the **ReArm protocol** (Muller et al., 2021), a randomized controlled trial at CHU Montpellier. Thirty post-stroke patients were assessed with standardized clinical tests.

We use `openpyxl` (cell-by-cell reading) rather than `pandas.read_excel` because the file has header formatting issues (extra spaces, merged cells) that cause pandas to misread columns.

**Variables loaded:**

- `SUBJID` — patient identifier
- `Sex`, `Age`, `Paresis` — demographic and clinical info
- `PANU` — Proximal Arm Non-Use (%)
- `Barthel` — Barthel Index of daily living independence (/100)
- `FMUE_TOTAL` — Fugl-Meyer Upper Extremity (/66)
- `WMFT_Total` — Wolf Motor Function Test (/75)
- `BBT_P` — Box & Block Test, paretic hand

In [2]:
# Open the Excel workbook
workbook = openpyxl.load_workbook('data/Outcomes_Rearm_01_04_23.xlsx')
sheet = workbook['V1']  # The file has only one sheet

# Read each row into a dictionary, then build a DataFrame
clinical_rows = []
for row_idx in range(2, sheet.max_row + 1):   # row 1 is headers, so start at row 2
    clinical_rows.append({
        'SUBJID':      sheet.cell(row_idx, 1).value,
        'Sex':         sheet.cell(row_idx, 3).value,
        'Age':         sheet.cell(row_idx, 4).value,
        'Paresis':     sheet.cell(row_idx, 6).value,
        'PANU':        sheet.cell(row_idx, 7).value,
        'Barthel':     sheet.cell(row_idx, 8).value,
        'FMUE_TOTAL':  sheet.cell(row_idx, 11).value,
        'WMFT_Total':  sheet.cell(row_idx, 12).value,
        'BBT_P':       sheet.cell(row_idx, 17).value,
    })

clinical_df = pd.DataFrame(clinical_rows)

# Standardize the patient ID so it matches the accelerometry file
# Clinical file uses 'C1-P01', accelerometry uses 'C1P01'
clinical_df['ID'] = clinical_df['SUBJID'].str.replace('-', '')

# Quick check
print(f"Clinical data loaded: {len(clinical_df)} patients")
clinical_df.head()

Clinical data loaded: 30 patients


,SUBJID,Sex,Age,Paresis,PANU,Barthel,FMUE_TOTAL,WMFT_Total,BBT_P,ID
0,C1-P01,Homme,63,Droite,3.0,85.0,45,64,38,C1P01
1,C1-P02,Homme,49,Gauche,3.0,100.0,65,73,52,C1P02
2,C1-P03,Homme,59,Gauche,2.0,90.0,48,56,11,C1P03
3,C1-P04,Homme,61,Gauche,1.0,95.0,60,62,27,C1P04
4,C1-P05,Homme,52,Gauche,3.0,90.0,57,57,15,C1P05
